# Simulacion Roland Garros

In [5]:
import random
import pickle
import pandas as pd

In [17]:
# WTA limpio como fuente de consulta para generar los features del partido a simular
df=pd.read_csv('wta_limpio.csv', parse_dates=['Date'])

In [ ]:
with open ('gbx_red.model', 'rb') as archivo_entrada:
    modeloML = pickle.load(archivo_entrada)
print(modeloML)

In [6]:
# Guardarlas en un archivo para importarlas directamente

def forma_reciente(df, jugadora, fecha_limite, meses=2):
    """Win rate de una jugadora en los N meses anteriores a la fecha.
    Devuelve 0 si no tiene partidos (lesión o pausa larga)"""
    fecha_inicio = fecha_limite - pd.DateOffset(months=meses)
    mask = (
        ((df['Player_1'] == jugadora) | (df['Player_2'] == jugadora)) &
        (df['Date'] < fecha_limite) &
        (df['Date'] >= fecha_inicio)
    )
    partidos = df[mask]
    if len(partidos) == 0:
        return 0
    victorias = (partidos['Winner'] == jugadora).sum()
    return victorias/len(partidos)
def winrate(df, jugadora, fecha_limite, superficie=None, ronda=None):
    """Win rate histórico de una jugadora, filtrable por superficie y/o ronda.
    Devuelve 0.4 si no tiene historial (ligeramente por debajo de neutro = novata en esa condición)"""
    mask = (
        ((df['Player_1'] == jugadora) | (df['Player_2'] == jugadora)) &
        (df['Date'] < fecha_limite)
    )
    if superficie:
        mask &= (df['Surface'] == superficie)
    if ronda:
        mask &= (df['Round'] == ronda)
    partidos = df[mask]
    if len(partidos) == 0:
        return 0.4
    victorias = (partidos['Winner'] == jugadora).sum()
    return victorias/len(partidos)
def headtohead(df, p1, p2, fecha_limite):
    """% de victorias de p1 sobre p2 en enfrentamientos directos previos a la fecha.
    Devuelve 0.5 si nunca se han enfrentado (neutro)"""
    mask = (
        (
            ((df['Player_1'] == p1) & (df['Player_2'] == p2)) |
            ((df['Player_1'] == p2) & (df['Player_2'] == p1))
        ) & (df['Date'] < fecha_limite)
    )
    partidos = df[mask]
    if len(partidos) == 0:
        return 0.5
    victorias_p1 = (partidos['Winner'] == p1).sum()
    return victorias_p1/len(partidos)
def experiencia(df, jugadora, fecha_limite):
    """Número total de partidos jugados por la jugadora antes de la fecha"""
    mask = (
        ((df['Player_1'] == jugadora) | (df['Player_2'] == jugadora)) &
        (df['Date'] < fecha_limite)
    )
    return df[mask].shape[0]

┌─────────────────────────────────────────┐
│           SIMULACIÓN MONTE CARLO        │  ← repite 10.000 veces
│                                         │
│   Cuartos → Semis → Final               │
│   para cada partido llama a...          │
│                                         │
│   ┌─────────────────────────────────┐   │
│   │         MODELO ML               │   │  ← entrenado con datos históricos
│   │  input: jugadora A vs B         │   │
│   │  output: probabilidad 0..1      │   │
│   └─────────────────────────────────┘   │
│                                         │
└─────────────────────────────────────────┘
         ↓ resultado final
   {"Swiatek": 38%, "Sabalenka": 22%, ...}

In [ ]:
# Se carga el cuadro. Se simulan los partidos y se obtienen las probabilidades de cada 

In [55]:
# Código de base para Monte Carlo
def simular_partido(prob_a):
  # lanzamos un dado cargado
  return random.random() < prob_a

def simular_torneo(df, cache_features, cache_h2h, cuadro, modelo):
    fecha = pd.to_datetime('2026-05-05')
    superficie = 'Clay'
    rondas = ['1st Round', '2nd Round', '3rd Round', '4th Round', 
              'Quarterfinals', 'Semifinals', 'The Final']
    
    jugadoras = cuadro.copy()
    indice_ronda = 3  # Octavos para el cuadro que estoy simulando

    while len(jugadoras) > 1:
        siguiente_ronda = []
        ronda_actual = rondas[indice_ronda]  # Nombre de la ronda actual
   
        # Emparejar jugadoras
        for i in range(0, len(jugadoras), 2):
            a, b = jugadoras[i], jugadoras[i+1]
            X = construir_features_cache (cache_features, cache_h2h, a, b, superficie, ronda_actual, fecha)
            prob_a = modelo.predict_proba(X)[0][1]  # Clase 1: probabilidad de que gane p1(a)
            
            ganadora = a if simular_partido(prob_a) else b
            siguiente_ronda.append(ganadora)
        
        # Actualizar para la siguiente ronda
        jugadoras = siguiente_ronda
        indice_ronda += 1  
        

    return jugadoras[0]

In [29]:

# SIMULACIÓN MONTE CARLO
def simulacion_montecarlo(df, cache_features, cacheh2h, cuadro, modelo, n_simulaciones=10000):

    victorias = {}  # Diccionario vacío
    
    for _ in range(n_simulaciones):
        campeona = simular_torneo(df, cache_features, cacheh2h, cuadro, modelo)

        if campeona in victorias:
            victorias[campeona] += 1
        else:
            victorias[campeona] = 1
    
    # Calcular probabilidades
    probabilidades = {}
    for jugadora, wins in victorias.items():
        probabilidades[jugadora] = wins / n_simulaciones
    
    return probabilidades, victorias

# Cuadro ficticio prueba
def crear_cuadro_top16():
    return [
        "Swiatek I.", "Sabalenka A.", "Gauff C.", "Rybakina E.",
        "Pegula J.", "Vondrousova", "Jabeur O.", "Zheng",
        "Sakkari M.", "J. Ostapenko", "Badosa P.", "D. Kasatkina",
        "Keys M.", "Azarenka V.", "Svitolina E.", "Navarro E."
    ]



In [ ]:
# Funciones para calcular features a fecha de partido.

def get_ranking(df_wta, jugadora, fecha_limite):
    mask = (
        ((df_wta['Player_1'] == jugadora) | (df_wta['Player_2'] == jugadora)) &
        (df_wta['Date'] < fecha_limite)
    )
    partidos = df_wta[mask].sort_values('Date', ascending=False)
    if len(partidos) == 0:
        return 500
    ultimo = partidos.iloc[0]
    if ultimo['Player_1'] == jugadora:
        return ultimo['Rank_1']
    return ultimo['Rank_2']



In [ ]:
# La simulación tarda muchísimo si tiene que calcular cada vez (10.000) todas las features. Como las jugadoras ya
# las sabemos, calculamos las features antes, las almacenamos y luego llamamos ahí en vez de a las funciones

In [ ]:
# def calculo_features_jugadoras_torneo (wta, cuadro):
#     # ANTES de la simulación — calcular features de cada jugadora una sola vez
#     fecha = pd.to_datetime('2026-05-05')
#     superficie = 'Clay'

#     cache_features = {}
#     for jugadora in cuadro:
#         cache_features[jugadora] = {
#             'wins2meses':       forma_reciente(wta, jugadora, fecha),
#             'ratio_superficie': winrate(wta, jugadora, fecha, superficie=superficie),
#             'experiencia':      experiencia(wta, jugadora, fecha),
#             'ranking':          get_ranking(wta, jugadora, fecha)
#         }

#     # Y una caché para los h2h entre cada par
#     cache_h2h = {}
#     for i, p1 in enumerate(cuadro):
#         for p2 in cuadro[i+1:]:
#             cache_h2h[(p1, p2)] = headtohead(wta, p1, p2, fecha)
#             cache_h2h[(p2, p1)] = 1 - cache_h2h[(p1, p2)]
    
#     return cache_features, cache_h2h

In [ ]:

def calculo_features_jugadoras_torneo (df, cuadro):
    fecha = pd.to_datetime('2026-05-05')
    superficie = 'Clay'
    rondas = ['1st Round', '2nd Round', '3rd Round', '4th Round', 
              'Quarterfinals', 'Semifinals', 'The Final']
    
    cache_features = {}
    for jugadora in cuadro:

        # Calcular los valores
        wins2meses_val = forma_reciente(df, jugadora, fecha)
        ratio_superficie_val = winrate(df, jugadora, fecha, superficie=superficie)
        experiencia_val = experiencia(df, jugadora, fecha)
        ranking_val = get_ranking(df, jugadora, fecha)
        
        # Calcular winrate por ronda
        winrate_por_ronda = {}
        for ronda in rondas:
            winrate_por_ronda[ronda] = winrate(df, jugadora, fecha, ronda=ronda)
    
        cache_features[jugadora] = {
            'wins2meses': wins2meses_val,
            'ratio_superficie': ratio_superficie_val,
            'experiencia': experiencia_val,
            'ranking': ranking_val,
            'winrate_por_ronda': winrate_por_ronda  
        }

    # Y una caché para los h2h entre cada par
    cache_h2h = {}
    for i, p1 in enumerate(cuadro):
        for p2 in cuadro[i+1:]:
            cache_h2h[(p1, p2)] = headtohead(df, p1, p2, fecha)
            cache_h2h[(p2, p1)] = 1 - cache_h2h[(p1, p2)]
    
    return cache_features, cache_h2h

In [52]:
# Da error
def construir_features_cache (cache_features, cache_h2h, p1, p2, superficie, ronda, fecha):
    f1 = cache_features[p1]
    f2 = cache_features[p2]

    row = {
        'surface':             superficie,
        'round':               ronda,
        'rank_diff':           f1['ranking'] - f2['ranking'],
        'wins2meses_p1':       f1['wins2meses'],
        'wins2meses_p2':       f2['wins2meses'],
        'ratio_superficie_p1': f1['ratio_superficie'],
        'ratio_superficie_p2': f2['ratio_superficie'],
        'h2h':                 cache_h2h.get((p1, p2), 0.5),
        # 'ratio_ronda_p1':      winrate(wta, p1, fecha, ronda=ronda),
        # 'ratio_ronda_p2':      winrate(wta, p2, fecha, ronda=ronda),
        'ratio_ronda_p1':      f1['winrate_por_ronda'][ronda],
        'ratio_ronda_p2':      f2['winrate_por_ronda'][ronda],
        'experiencia_p1':      f1['experiencia'],
        'experiencia_p2':      f2['experiencia'],
        'tournament_type':     'GS'
    }
    return pd.DataFrame([row])

In [56]:
# Ejecutar simulación
cuadro = crear_cuadro_top16()
cache_features, cache_h2h = calculo_features_jugadoras_torneo (df, cuadro)


In [57]:
probabilidades, victorias = simulacion_montecarlo(df, cache_features, cache_h2h, cuadro, modeloML, 5)

# Mostrar resultados
print("Resultados después de 10,000 simulaciones:")
print("-" * 40)
for jugadora, prob in sorted(probabilidades.items(), key=lambda x: x[1], reverse=True):
    print(f"{jugadora}: {prob:.2%} ({victorias[jugadora]} títulos)")

Resultados después de 10,000 simulaciones:
----------------------------------------
Sabalenka A.: 40.00% (2 títulos)
Svitolina E.: 20.00% (1 títulos)
Badosa P.: 20.00% (1 títulos)
Swiatek I.: 20.00% (1 títulos)


In [58]:
import time

# Mide el tiempo para 100 simulaciones
inicio = time.time()
probabilidades, victorias = simulacion_montecarlo(df, cache_features, cache_h2h, 
                                                   cuadro, modeloML, 100)
fin = time.time()

tiempo_100 = fin - inicio
print(f"100 simulaciones tardaron: {tiempo_100:.2f} segundos")
print(f"Estimación para 10,000 simulaciones: {tiempo_100 * 100:.2f} segundos = {(tiempo_100 * 100)/60:.1f} minutos")
print("Resultados después de 10,000 simulaciones:")
print("-" * 40)
for jugadora, prob in sorted(probabilidades.items(), key=lambda x: x[1], reverse=True):
    print(f"{jugadora}: {prob:.2%} ({victorias[jugadora]} títulos)")

100 simulaciones tardaron: 4.94 segundos
Estimación para 10,000 simulaciones: 494.42 segundos = 8.2 minutos
Resultados después de 10,000 simulaciones:
----------------------------------------
Swiatek I.: 21.00% (21 títulos)
Pegula J.: 18.00% (18 títulos)
Gauff C.: 12.00% (12 títulos)
Sabalenka A.: 10.00% (10 títulos)
Svitolina E.: 10.00% (10 títulos)
Jabeur O.: 7.00% (7 títulos)
Keys M.: 6.00% (6 títulos)
Sakkari M.: 6.00% (6 títulos)
Rybakina E.: 6.00% (6 títulos)
Badosa P.: 2.00% (2 títulos)
Azarenka V.: 1.00% (1 títulos)
Navarro E.: 1.00% (1 títulos)
